# Test: Separate Basis Set Utilities

This notebook tests the `separate_basis_set` module, which extracts the difference between a full basis set and a base basis set.
The workflow demonstrates:
1. Downloading basis sets from BSE
2. Parsing them into dictionaries
3. Computing the difference (augmented primitives)
4. Formatting and saving the results

In [ ]:
from pprint import pprint

from deepdiff import DeepDiff
import basis_set_exchange as bse

import separate_basis_set as sbs

## Setup: Import Required Libraries and Modules

In [ ]:
# Download base and augmented basis sets from BSE
data1 = bse.get_basis("cc-pvtz-dk", fmt='psi4', header=True, optimize_general=True)
data2 = bse.get_basis("aug-cc-pvtz-dk", fmt='psi4', header=True, optimize_general=True)

## Step 1: Download Basis Sets from Basis Set Exchange
Download two basis sets: a base set (cc-pvtz-dk) and a full augmented set (aug-cc-pvtz-dk) for comparison.

In [ ]:
# Parse basis sets into dictionary structures and extract the difference
dict1, header = sbs.get_basis_set_dict(data1)
dict2, header = sbs.get_basis_set_dict(data2)
new_dict = sbs.subtract_dict_contents(dict2, dict1)

## Step 2: Parse and Compare Basis Sets
Parse both basis sets into nested dictionaries, then compute the difference to extract only the augmented primitives.

In [ ]:
# Display the augmented primitives for Helium as a sample
pprint(new_dict["He"])

['0',
 [[['S', '1', '1.00'], [['0.0513800', '1.0000000']]],
  [['P', '1', '1.00'], [['0.1993000', '1.0000000']]],
  [['D', '1', '1.00'], [['0.4592000', '1.0000000']]]]]


## Step 3: Inspect Results
Display the augmented primitives for a sample element (Helium) to verify the extraction worked correctly.

In [ ]:
# Print the augmented basis set in Psi4 .gbs format
print(sbs.form_basis_set_file(new_dict))

H    0
S     1    1.00
  2.5260000E-02  1.0000000E+00
P     1    1.00
  1.0200000E-01  1.0000000E+00
D     1    1.00
  2.4700000E-01  1.0000000E+00
****
He   0
S     1    1.00
  5.1380000E-02  1.0000000E+00
P     1    1.00
  1.9930000E-01  1.0000000E+00
D     1    1.00
  4.5920000E-01  1.0000000E+00
****
Li   0
S     1    1.00
  7.6000000E-03  1.0000000E+00
P     1    1.00
  9.1000000E-03  1.0000000E+00
D     1    1.00
  3.7100000E-02  1.0000000E+00
F     1    1.00
  8.1600000E-02  1.0000000E+00
****
Be   0
S     1    1.00
  1.5030000E-02  1.0000000E+00
P     1    1.00
  7.0600000E-03  1.0000000E+00
D     1    1.00
  6.5400000E-02  1.0000000E+00
F     1    1.00
  1.5330000E-01  1.0000000E+00
****
B    0
S     1    1.00
  2.9140000E-02  1.0000000E+00
P     1    1.00
  2.0960000E-02  1.0000000E+00
D     1    1.00
  6.0400000E-02  1.0000000E+00
F     1    1.00
  1.6300000E-01  1.0000000E+00
****
C    0
S     1    1.00
  4.4020000E-02  1.0000000E+00
P     1    1.00
  3.5690000E-02  1.00000

## Step 5: Test the Main Function
Call `separate_basis_functions()` which should produce the same result as the manual steps above.

In [ ]:
# Run the main function: downloads both basis sets, extracts difference, and saves all three .gbs files
sbs.separate_basis_functions("aug-cc-pvtz-dk", "cc-pvtz-dk")

['spherical', '', '!----------------------------------------------------------------------', '! Basis Set Exchange', '! Version 0.10', '! https://www.basissetexchange.org', '!----------------------------------------------------------------------', '!   Basis set: aug-cc-pVTZ-DK', '! Description: VTZ2P   All-electron Douglas-Kroll Valence Triple Zeta +', '!                    Polarization', '!        Role: orbital', '!     Version: 0  (Data from the original Basis Set Exchange)', '!----------------------------------------------------------------------', '', '', '****']


## Step 6: Verify Results
Read the output file from `separate_basis_functions()` and compare it to the manually extracted result to verify correctness.

In [ ]:
# Read the output file created by separate_basis_functions()
with open("stripped-aug-cc-pvtz-dk.gbs", "r") as f:
    output_data = f.read()

# Parse the output file back into a dictionary
output_dict, _ = sbs.get_basis_set_dict(output_data)

# Compare: format both results and check if they're identical
manual_output = sbs.form_basis_set_file(new_dict)
function_output = sbs.form_basis_set_file(output_dict)

# Check if results match
if manual_output == function_output:
    print("PASSED: Manual extraction matches separate_basis_functions() output")
else:
    print("FAILED: Outputs differ")
    diff = DeepDiff(new_dict, output_dict, verbose_level=2)
    print(f"Differences:\n{diff}")

## Step 7: Round-trip Formatting Test
Verify output file formatting consistency by loading a basis set file, parsing it, and reformatting it.

In [ ]:
# Parse and reformat: verify the formatting functions produce consistent output
data_dict, header = sbs.get_basis_set_dict(data)
data = "\n".join(header) + "\n" + sbs.form_basis_set_file(data_dict) # Round trip for formatting
open("basis-cc-pvtz-dk.gbs", 'w').write(data)

311657